# Tamil Nadu 2026 Assembly Election – Analysis Notebook
### RPC: Decoding the 2026 Tamil Nadu Assembly Election
**Questions covered:** Q1 Geographic Story · Q2 Flip Story · Q6 Margin of Victory

*Data sources: tn_2021_results.csv · tn_2026_results.csv · constituency_master.csv*


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings; warnings.filterwarnings('ignore')

df21 = pd.read_csv('tn_2021_results.csv')
df26 = pd.read_csv('tn_2026_results.csv')
master = pd.read_csv('constituency_master.csv')

print("2021 rows:", len(df21), "| Constituencies:", df21['ac_number'].nunique())
print("2026 rows:", len(df26), "| Constituencies:", df26['ac_number'].nunique())
print("Regions:", sorted(df21['region'].unique()))


In [ ]:
# ── Helper functions ────────────────────────────────────────────────────────
def get_winners(df, year):
    winners = (df.sort_values('votes', ascending=False)
                 .groupby('ac_number', as_index=False).first())
    ranked = df.sort_values(['ac_number','votes'], ascending=[True,False])
    runners = (ranked.groupby('ac_number').nth(1)
                     [['ac_number','votes']]
                     .rename(columns={'votes':'runner_votes'})
                     .reset_index(drop=True))
    total = df.groupby('ac_number')['votes'].sum().reset_index(name='total_votes')
    w = winners.merge(runners, on='ac_number').merge(total, on='ac_number')
    w['vote_share'] = (w['votes'] / w['total_votes'] * 100).round(2)
    w['margin'] = w['votes'] - w['runner_votes']
    w['year'] = year
    return w

def classify_party(p):
    return p if p in ['DMK','AIADMK','TVK','NTK'] else 'Others'

def classify_share(s):
    if s > 50:  return '>50% dominant'
    elif s < 35: return '<35% fragmented'
    else:        return '35-50% competitive'

w21 = get_winners(df21, 2021)
w26 = get_winners(df26, 2026)
w21['party_grp'] = w21['party'].apply(classify_party)
w26['party_grp'] = w26['party'].apply(classify_party)
print("Winners extracted — 2021:", len(w21), "| 2026:", len(w26))


---
## Q1 – The Geographic Story
> How did seat distribution shift across Tamil Nadu's six regions between 2021 and 2026?

In [ ]:
# Q1: Seat counts by region and party
seats21 = w21.groupby(['region','party_grp']).size().reset_index(name='seats').assign(year=2021)
seats26 = w26.groupby(['region','party_grp']).size().reset_index(name='seats').assign(year=2026)
all_seats = pd.concat([seats21, seats26], ignore_index=True)

pivot = all_seats.pivot_table(index=['region','party_grp'], columns='year',
                               values='seats', fill_value=0).reset_index()
pivot.columns = ['region','party','seats_2021','seats_2026']
pivot['change'] = pivot['seats_2026'] - pivot['seats_2021']
print(pivot.sort_values(['region','party']).to_string(index=False))


In [ ]:
# Q1: Grouped bar chart
color_map = {'DMK':'#E24B4A','AIADMK':'#378ADD','TVK':'#1D9E75','NTK':'#EF9F27','Others':'#888780'}
fig = px.bar(all_seats, x='region', y='seats', color='party_grp',
             facet_col='year', barmode='group',
             color_discrete_map=color_map,
             labels={'seats':'Seats won','party_grp':'Party','region':'Region'},
             title='Q1: Seat distribution by region — 2021 vs 2026', height=450)
fig.show()
fig.write_html('q1_regional_seats.html')
print("Saved: q1_regional_seats.html")


---
## Q2 – The Flip Story
> In how many constituencies did the winning party change between 2021 and 2026?

In [ ]:
# Q2: Build flip table
w21r = w21[['ac_number','constituency','party_grp','region','reserved']].rename(columns={'party_grp':'winner_2021'})
w26r = w26[['ac_number','party_grp']].rename(columns={'party_grp':'winner_2026'})
flips = w21r.merge(w26r, on='ac_number')
flips['flipped'] = flips['winner_2021'] != flips['winner_2026']

total = len(flips)
n_flipped = flips['flipped'].sum()
print(f"Total constituencies: {total}")
print(f"Flipped: {n_flipped} ({n_flipped/total*100:.1f}%)")
print(f"Retained: {total - n_flipped} ({(total-n_flipped)/total*100:.1f}%)")

sankey_df = (flips.groupby(['winner_2021','winner_2026'])
                  .size().reset_index(name='count')
                  .sort_values('count', ascending=False))
print("\nTop flows:")
print(sankey_df.head(8).to_string(index=False))


In [ ]:
# Q2: Sankey diagram
parties = ['DMK','AIADMK','TVK','NTK','Others']
src_labels = [p+' 2021' for p in parties]
tgt_labels = [p+' 2026' for p in parties]
all_labels = src_labels + tgt_labels
node_colors = {'DMK 2021':'#E24B4A','DMK 2026':'#E24B4A','AIADMK 2021':'#378ADD',
               'AIADMK 2026':'#378ADD','TVK 2021':'#1D9E75','TVK 2026':'#1D9E75',
               'NTK 2021':'#EF9F27','NTK 2026':'#EF9F27','Others 2021':'#888780','Others 2026':'#888780'}
colors = [node_colors[l] for l in all_labels]
label_idx = {l:i for i,l in enumerate(all_labels)}
sources, targets, values = [], [], []
for _, row in sankey_df.iterrows():
    sk = row['winner_2021']+' 2021'
    tk = row['winner_2026']+' 2026'
    if sk in label_idx and tk in label_idx:
        sources.append(label_idx[sk]); targets.append(label_idx[tk]); values.append(row['count'])
fig = go.Figure(go.Sankey(
    node=dict(label=all_labels, color=colors, pad=20, thickness=24),
    link=dict(source=sources, target=targets, value=values, color='rgba(150,150,150,0.3)')
))
fig.update_layout(title_text='Q2: Seat flow — 2021 to 2026 (Sankey)', height=500)
fig.show()
fig.write_html('q2_sankey.html')
print("Saved: q2_sankey.html")


In [ ]:
# Q2: Flips by region
flip_region = (flips.groupby('region')
                    .agg(total=('ac_number','count'), flipped=('flipped','sum'))
                    .reset_index())
flip_region['flip_pct'] = (flip_region['flipped'] / flip_region['total'] * 100).round(1)
print(flip_region.sort_values('flip_pct', ascending=False).to_string(index=False))

fig = px.bar(flip_region.sort_values('flip_pct', ascending=False),
             x='region', y='flip_pct', color='flip_pct',
             color_continuous_scale='RdYlGn_r', text='flipped',
             title='Q2: Percentage of constituencies that flipped, by region', height=400)
fig.update_traces(texttemplate='%{text} seats', textposition='outside')
fig.show()
fig.write_html('q2_flip_region.html')


---
## Q6 – The Margin of Victory Story
> How did the average margin of victory change? How many won with >50% or <35% of votes?

In [ ]:
# Q6: Margin analysis
w21['share_bucket'] = w21['vote_share'].apply(classify_share)
w26['share_bucket'] = w26['vote_share'].apply(classify_share)

print("=== 2021 ===")
print(w21['share_bucket'].value_counts())
print(f"Average margin: {w21['margin'].mean():,.0f} votes")
print()
print("=== 2026 ===")
print(w26['share_bucket'].value_counts())
print(f"Average margin: {w26['margin'].mean():,.0f} votes")
print()
change = w26['margin'].mean() - w21['margin'].mean()
print(f"Change in avg margin: {change:+,.0f} votes ({change/w21['margin'].mean()*100:+.1f}%)")


In [ ]:
# Q6: Distribution histogram
all_margins = pd.concat([w21, w26], ignore_index=True)
all_margins['year'] = all_margins['year'].astype(str)
fig = px.histogram(all_margins, x='vote_share', color='year', barmode='overlay',
                   nbins=30, opacity=0.75,
                   color_discrete_map={'2021':'#378ADD','2026':'#1D9E75'},
                   labels={'vote_share':'Winner vote share (%)','year':'Year'},
                   title='Q6: Distribution of winner vote share — 2021 vs 2026', height=420)
fig.add_vline(x=35, line_dash='dash', line_color='#E24B4A',
              annotation_text='35% threshold', annotation_position='top right')
fig.add_vline(x=50, line_dash='dash', line_color='#EF9F27',
              annotation_text='50% threshold', annotation_position='top right')
fig.show()
fig.write_html('q6_margin_dist.html')
print("Saved: q6_margin_dist.html")


In [ ]:
# Q6: Bucket comparison
b21 = w21['share_bucket'].value_counts().reset_index()
b21.columns = ['bucket','count']; b21['year'] = '2021'
b26 = w26['share_bucket'].value_counts().reset_index()
b26.columns = ['bucket','count']; b26['year'] = '2026'
buckets = pd.concat([b21, b26])
fig = px.bar(buckets, x='bucket', y='count', color='year', barmode='group',
             color_discrete_map={'2021':'#378ADD','2026':'#1D9E75'},
             title='Q6: How dominant were winners? — 2021 vs 2026',
             labels={'count':'Constituencies','bucket':'Vote share range'},
             text='count', height=420)
fig.update_traces(textposition='outside')
fig.show()
fig.write_html('q6_buckets.html')

# Scatter
fig2 = px.scatter(w26, x='vote_share', y='margin', color='party_grp',
                  color_discrete_map={'DMK':'#E24B4A','AIADMK':'#378ADD',
                                      'TVK':'#1D9E75','NTK':'#EF9F27','Others':'#888780'},
                  hover_data=['constituency','party'],
                  labels={'vote_share':'Winner vote share (%)','margin':'Margin (votes)','party_grp':'Party'},
                  title='Q6: Vote share vs margin — 2026', height=460)
fig2.add_vline(x=35, line_dash='dash', line_color='#E24B4A', opacity=0.4)
fig2.add_vline(x=50, line_dash='dash', line_color='#EF9F27', opacity=0.4)
fig2.show()
fig2.write_html('q6_scatter.html')
print("Saved: q6_buckets.html, q6_scatter.html")


In [ ]:
# ── Export all CSVs for Power BI ────────────────────────────────────────────
all_seats.to_csv('q1_region_seats.csv', index=False)
flips.to_csv('q2_flips_detail.csv', index=False)
sankey_df.to_csv('q2_sankey_flows.csv', index=False)
all_margins[['ac_number','constituency','party','party_grp','region','reserved',
             'votes','total_votes','vote_share','margin','share_bucket','year']].to_csv('q6_margins.csv', index=False)
pd.concat([w21, w26], ignore_index=True).to_csv('winners_all.csv', index=False)
print("Exports complete:")
for f in ['q1_region_seats.csv','q2_flips_detail.csv','q2_sankey_flows.csv','q6_margins.csv','winners_all.csv']:
    print(" ", f)


---
## Key findings summary

| Question | Finding |
|---|---|
| Q1 Geographic | TVK won 108 seats — dominant in Chennai Metro (29/32) and Kongu (16/33). DMK fell from 133 to 59. |
| Q2 Flip story | 161 of 234 constituencies (69%) changed hands — 65 seats moved directly from DMK to TVK. |
| Q6 Margin | Average margin dropped from 22,871 to 16,784 votes (−27%). Seats won with <35% jumped from 2 to 64. |

> **Data limitation note:** 2026 turnout column is blank in tn_2026_results.csv.
> Source 2026 turnout from ECI Form-20 at results.eci.gov.in for any turnout analysis.
